<a href="https://colab.research.google.com/github/sriharan17/SriflyrankAI/blob/main/work/notebooks/w04_baseline_score.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sriharan17/SriflyrankAI/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

My Rule

Prioritize content for refresh when it is stale and has a low CTR compared with its search position. Higher staleness and weaker CTR relative to position should receive a higher priority score.

Reason Codes

- STALE_LOW_CTR — Content is highly stale and has weak CTR for its position → REFRESH
- STALE_CONTENT — Content is highly stale → REFRESH
- LOW_CTR_POSITION — CTR is low compared with the page's position → OPTIMIZE_CTR
- NO_PRIORITY_SIGNAL — Neither signal indicates a strong problem → MONITOR

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [5]:
import pandas as pd
import os

df = pd.read_csv("/content/content_refresh_anonymized.csv")

print("Rows:", len(df))
print("Columns:")
print(df.columns.tolist())

display(df.head())

Rows: 30000
Columns:
['content_id', 'client_id', 'search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count', 'provider_used', 'model_used', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier', 'age_tier_order', 'days_since_last_update', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'impression_tier', 'position_tier', 'trend_direction', 'trend_pct']


,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,char_count_tier,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,...,15000-25000,0.76,10.6,5.88,4.55,0.0,good,striking,down,-41.4
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,...,15000-25000,0.05,20.3,0.00,10.00,0.0,good,page_3_5,down,-57.7
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,...,15000-25000,0.09,36.5,0.00,28.57,0.0,good,page_3_5,down,-60.9
3,content_331d6c4de07b,client_19581e27de,10.0,0.00,LOW,0.00,keyword article,commercial,NaN,NaN,...,NaN,0.49,6.2,1.28,3.45,0.0,good,page_1,stable,-13.8
4,content_d99b7a2d90ca,client_3fdba35f04,0.0,0.00,LOW,0.00,keyword article,informational,2803.0,17469.0,...,15000-25000,0.13,44.0,0.00,24.29,0.0,good,page_3_5,down,-34.7


In [9]:
import numpy as np

# Ensure necessary columns are numeric and handle potential NaNs
df['days_since_last_update'] = pd.to_numeric(df['days_since_last_update'], errors='coerce').fillna(0)
df['ctr'] = pd.to_numeric(df['ctr'], errors='coerce').fillna(0)
df['avg_position'] = pd.to_numeric(df['avg_position'], errors='coerce').fillna(0)

# Calculate dynamic thresholds based on data distribution
# For staleness: higher values are worse. Using 80th percentile for 'highly stale'.
stale_threshold_days = df['days_since_last_update'].quantile(0.80)

# For CTR: lower values are worse. Using 20th percentile for 'weak CTR'.
low_ctr_threshold = df['ctr'].quantile(0.20)

# For average position: lower values are better. Using median to identify 'good position' (i.e., better than average).
good_position_threshold = df['avg_position'].median()

# Initialize new columns for reason code, action, and score
df['reason_code'] = 'NO_PRIORITY_SIGNAL'
df['action'] = 'MONITOR'
df['score'] = 0.0

# --- Apply rules and assign scores based on defined priority ---

# Rule 1: STALE_LOW_CTR - highest priority, combines high staleness and weak CTR for good positions
condition_stale_low_ctr = (
    (df['days_since_last_update'] > stale_threshold_days) &
    (df['ctr'] < low_ctr_threshold) &
    (df['avg_position'] < good_position_threshold)
)

df.loc[condition_stale_low_ctr, 'reason_code'] = 'STALE_LOW_CTR'
df.loc[condition_stale_low_ctr, 'action'] = 'REFRESH'

# Score for STALE_LOW_CTR: combination of normalized staleness and inverted CTR. Higher values mean worse.
max_days = df['days_since_last_update'].max()
norm_stale = df['days_since_last_update'] / max_days if max_days > 0 else 0

max_ctr = df['ctr'].max()
min_ctr = df['ctr'].min()
if (max_ctr - min_ctr) > 0:
    inv_norm_ctr = (df['ctr'].max() - df['ctr']) / (df['ctr'].max() - df['ctr'].min())
else:
    inv_norm_ctr = 0 # All CTR values are the same, so no difference in inverse normalized CTR

df.loc[condition_stale_low_ctr, 'score'] = (norm_stale * 50 + inv_norm_ctr * 50).fillna(0)


# Rule 2: STALE_CONTENT - high staleness, but not meeting STALE_LOW_CTR conditions
condition_stale = (df['days_since_last_update'] > stale_threshold_days) & (~condition_stale_low_ctr)
df.loc[condition_stale, 'reason_code'] = 'STALE_CONTENT'
df.loc[condition_stale, 'action'] = 'REFRESH'
# Score for STALE_CONTENT: based on normalized staleness, with a slightly lower maximum ceiling
df.loc[condition_stale, 'score'] = (norm_stale * 70).fillna(0)


# Rule 3: LOW_CTR_POSITION - weak CTR for its position, but not meeting STALE_LOW_CTR or STALE_CONTENT conditions
condition_low_ctr_pos = (
    (df['ctr'] < low_ctr_threshold) &
    (df['avg_position'] < good_position_threshold) &
    (~condition_stale_low_ctr) &
    (~condition_stale)
)
df.loc[condition_low_ctr_pos, 'reason_code'] = 'LOW_CTR_POSITION'
df.loc[condition_low_ctr_pos, 'action'] = 'OPTIMIZE_CTR'
# Score for LOW_CTR_POSITION: based on inverted normalized CTR, with a slightly lower maximum ceiling
df.loc[condition_low_ctr_pos, 'score'] = (inv_norm_ctr * 60).fillna(0)


# Rule 4: NO_PRIORITY_SIGNAL - default, score remains 0.0

# Ensure scores are integers as they are used for confidence notes (e.g., x >= 75)
df['score'] = df['score'].astype(int)

# Create the ranked DataFrame by sorting by score in descending order
competition_level_df = df.sort_values(by='score', ascending=False).reset_index(drop=True)
competition_level_df['rank'] = competition_level_df.index + 1

# Output the top 5 rows of the ranked DataFrame for review
print("Top 5 rows of the ranked queue:")
display(competition_level_df[['rank', 'content_id', 'days_since_last_update', 'ctr', 'avg_position', 'reason_code', 'action', 'score']].head())

# Write the ranked DataFrame to CSV as specified in the notebook section hint
output_dir = 'work/outputs'
output_filename = os.path.join(output_dir, 'baseline_action_score.csv')
os.makedirs(output_dir, exist_ok=True) # Ensure the output directory exists
competition_level_df.to_csv(output_filename, index=False)

print(f"\nRanked queue saved to {output_filename}")

Top 5 rows of the ranked queue:


,rank,content_id,days_since_last_update,ctr,avg_position,reason_code,action,score
0,1,content_f6fdf87348f6,373,0.0,32.5,STALE_CONTENT,REFRESH,70
1,2,content_55a5b1c46474,373,0.0,7.5,STALE_CONTENT,REFRESH,70
2,3,content_3f3576c295f5,373,100.0,1.0,STALE_CONTENT,REFRESH,70
3,4,content_8d56efff1e71,372,0.0,35.0,STALE_CONTENT,REFRESH,69
4,5,content_1b4ec72dafd4,372,0.0,7.0,STALE_CONTENT,REFRESH,69



Ranked queue saved to work/outputs/baseline_action_score.csv


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [11]:
top20 = competition_level_df.head(20).copy()

top20["confidence_note"] = top20["score"].apply(
    lambda x: "High confidence" if x >= 75
    else ("Medium confidence" if x >= 50 else "Low confidence")
)

top20["what_would_make_it_wrong"] = (
    "The signal may be misleading, the data may be stale, "
    "or the page may already be performing well."
)

display(
    top20[
        ["rank", "score", "reason_code", "action",
         "confidence_note", "what_would_make_it_wrong"]
    ]
)


,rank,score,reason_code,action,confidence_note,what_would_make_it_wrong
0,1,70,STALE_CONTENT,REFRESH,Medium confidence,"The signal may be misleading, the data may be ..."
1,2,70,STALE_CONTENT,REFRESH,Medium confidence,"The signal may be misleading, the data may be ..."
2,3,70,STALE_CONTENT,REFRESH,Medium confidence,"The signal may be misleading, the data may be ..."
3,4,69,STALE_CONTENT,REFRESH,Medium confidence,"The signal may be misleading, the data may be ..."
4,5,69,STALE_CONTENT,REFRESH,Medium confidence,"The signal may be misleading, the data may be ..."
5,6,62,STALE_CONTENT,REFRESH,Medium confidence,"The signal may be misleading, the data may be ..."
6,7,62,STALE_CONTENT,REFRESH,Medium confidence,"The signal may be misleading, the data may be ..."
7,8,62,STALE_CONTENT,REFRESH,Medium confidence,"The signal may be misleading, the data may be ..."
8,9,58,STALE_CONTENT,REFRESH,Medium confidence,"The signal may be misleading, the data may be ..."
9,10,58,STALE_CONTENT,REFRESH,Medium confidence,"The signal may be misleading, the data may be ..."


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

In [12]:
# 4. Weak picks + leakage check

# --- Weak picks: flag anything low confidence or where reason/action mismatch ---
weak_picks = top20[top20["confidence_note"] == "Low confidence"]
print(f"Weak picks (low confidence): {len(weak_picks)}")
display(weak_picks[["rank", "score", "reason_code", "action", "confidence_note"]])

# --- Leakage check 1: no target/outcome columns snuck into the feature set ---
suspicious_terms = ["flag", "converted", "label", "target", "outcome", "result"]
leaked_cols = [
    c for c in competition_level_df.columns
    if any(term in c.lower() for term in suspicious_terms)
]
print("Columns that look like they could leak the outcome:", leaked_cols)

# --- Leakage check 2: no feature/date column is later than the scoring cutoff ---
date_cols = [c for c in competition_level_df.columns if "date" in c.lower()]
for col in date_cols:
    max_date = competition_level_df[col].max()
    print(f"{col}: latest value = {max_date}")
    # Manually confirm this is <= the date the score was computed as of.

# --- Summary check ---
print("\nSelf-check:")
print(f"- Suspicious/leaky columns found: {len(leaked_cols)} -> {'FAIL' if leaked_cols else 'PASS'}")
print(f"- Date columns to manually verify against scoring cutoff: {date_cols}")

Weak picks (low confidence): 0


,rank,score,reason_code,action,confidence_note


Columns that look like they could leak the outcome: []
days_since_last_update: latest value = 373

Self-check:
- Suspicious/leaky columns found: 0 -> PASS
- Date columns to manually verify against scoring cutoff: ['days_since_last_update']


In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.